In [ ]:
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
import os



loader = PyMuPDFLoader(f'contract_pdfs/')

docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""],  # more fallback levels
)

docs = splitter.split_documents(docs)
ids = list(map(lambda x: x.metadata['page'],docs))


embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_type="mmr",
    search_kwargs={'k': 3})


# query = "Organization decides to pay XXX dollars as per agreement for the shipment"
query = "Contact Information"
rel_docs = retriever.get_relevant_documents(query=query)



with open('output.txt', 'w') as f:
    f.write('\n'.join([doc.page_content for doc in rel_docs]))

In [ ]:
list(filter(lambda x: x.endswith('.pdf'), os.listdir('contract_pdfs')))
os.listdir('contract_pdfs')

['__pycache__',
 'main.py',
 'test_1.ipynb',
 'contract_pdfs',
 'LangchainGeminiLoader.py',
 'output.txt']

In [149]:
from LangchainGeminiLoader import get_llm
from langchain.prompts import PromptTemplate
llm = get_llm()

llm.invoke("""You are Querying a vectorstore which has Sentence Transformer Embedding and has Contractual Data Between Parties
           Query the vectorstore to get the Contact Information of parties
           Give me only the query in the response
           """)

AIMessage(content='"Retrieve all documents containing contact information, including names, addresses, phone numbers, and email addresses, for all parties involved in the contracts stored in this vectorstore."', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--db126689-fe65-47ec-8064-cbac820507a2-0', usage_metadata={'input_tokens': 44, 'output_tokens': 34, 'total_tokens': 78, 'input_token_details': {'cache_read': 0}})

In [ ]:
AlliedEsportsEntertainmentInc.txt
BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.txt
FreezeTagInc.txt
MEDIWOUNDLTD_01_15_2014-EX-10.txt
LohaCompanyltd_20191209_F-1_EX-10.txt

In [ ]:
import tiktoken
from langchain.prompts import PromptTemplate
from LangchainGeminiLoader import get_llm
tokens_needed = []
with open('output/AlliedEsportsEntertainmentInc.txt','r') as f:
    txt = f.read()
tokens = len(txt)/ 4
tokens_needed.append(tokens)
sum(tokens_needed)
# llm = get_llm()
prompt = PromptTemplate.from_template("""
You are an contract analyzing assistant.
1. You are required to extract the following data from the given Context.
- Orgnaizations Involved.
- Contract Agreement Summary.
- Compliance and Expectations Summary.
- Amount Deliverables for shipments in Agreement context.
- Contract Start & end dates.
- Termination & Expiry Information Summary.
Context: {context}

**Output Format**
1. Format the response in the json format in given schema.
### JSON Schema
{{
    "organizations_involved": [
        {{
        "name": "Organization Name",
        "email": "contact@example.com",
        "phone": "+1-555-123-4567"
        }}
    ],
    "contract_agreement_summary": [
        "Summarize the agreement in short bullet points.",
    ],
    "compliance_and_expectations_summary": [
        "Summarize all compliance obligations and expectations in separate statements.",
    ],
    "amount_deliverables_for_shipments": [
        {{
        "amount": "e.g. 1000 units",
        "period": "e.g. Monthly",
        "description": "Describe what is being delivered and under what terms."
        }}
    ],
    "contract_start_date": "YYYY-MM-DD",
    "contract_end_date": "YYYY-MM-DD",
    "termination_and_expiry_summary": [
        "List the key conditions or clauses under which the contract can be terminated.",
        "Include expiry rules or notice periods if stated."
    ]
}}
# Note
- Output only a json object. Never return any code block or any other format.
- Don't add any marker like ```json, ``` etc
- Don't output any extra character other than proper json
- Ensure all variations of duplicate data are preserved with contextual clarity.
- The merging process must account for nested structures and hierarchy.
- Further processing includes building knowledge graph from this output json. Hence the node and relationship is important
""")
llm = get_llm()
prompt = prompt.invoke({"context": txt})
token_count = len(prompt.to_string())/4
resp = llm.invoke(prompt)

print(f'Token Count: {resp.usage_metadata.get('total_tokens')}')
print(f'Response: {resp.content}')
    

Token Count: 2097
Response: ```json
{
    "organizations_involved": [
        {
            "name": "Disclosing Party",
            "email": null,
            "phone": null
        },
        {
            "name": "Receiving Party",
            "email": null,
            "phone": null
        },
        {
            "name": "Newegg",
            "email": null,
            "phone": null
        }
    ],
    "contract_agreement_summary": [
        "The Disclosing Party is sharing confidential information with the Receiving Party.",
        "Confidential Information includes technical, marketing, financial, employee, planning, and other proprietary information.",
        "Newegg's Confidential Information includes information and materials provided by Newegg in connection with this Agreement.",
        "The Receiving Party must protect the Confidential Information and only use it for the purposes of the agreement.",
        "The Receiving Party can only disclose Confidential Information 

In [38]:
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS, Qdrant
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
import os, shutil
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
import uuid


folder = 'output/'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
    except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))

pdfs = list(filter(lambda x: x.endswith('.pdf'), os.listdir('contract_pdfs')))
# pdfs = [
#     'BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.pdf', 
#     'FUSIONPHARMACEUTICALSINC_06_05_2020-EX-10.17-Supply Agreement - FUSION.pdf',
#     'MEDIWOUNDLTD_01_15_2014-EX-10.6-SUPPLY AGREEMENT copy.pdf',
#     'FreezeTagInc.pdf'
# ]

for pdf_name in pdfs:

    loader = PyMuPDFLoader(f'contract_pdfs/{pdf_name}')

    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=300,
        length_function=len
    )

    docs = splitter.split_documents(docs)

    ids = [str(uuid.uuid4()) for _ in range(len(docs))]

    embeddings = SentenceTransformerEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    # vectorstore = FAISS.from_documents(
    #     documents=docs,
    #     embedding=embeddings,
    # )
    qdrant = Qdrant.from_documents(
        documents=docs,
        ids=ids,
        embedding=embeddings,
        url="http://localhost:6333",  # Local Qdrant instance
        prefer_grpc=False,
        collection_name=pdf_name,
    )

    # qdrant.similarity_search(query="Contact Information of Sponsors and Orgnaizations")
    retriever = qdrant.as_retriever(search_type="mmr", search_kwargs={'k': 3})
    # query = "Find - Email Contact By Address Tel Fax Signature Phn Name Sponsors Organization Club"
    contact_info_docs = retriever.get_relevant_documents(query="Find - Email Contact By Address Tel Fax Signature Phn Name Sponsors Organization Club")
    transaction_docs = retriever.get_relevant_documents(query="Find - Transaction Details in XXX $ dollars or % percentages obligations of agreement.")
    # Combine contact_info_docs and transaction_docs
    combined_docs = contact_info_docs + transaction_docs

    # Remove duplicates based on page content
    unique_docs = {doc.metadata['_id']: doc for doc in combined_docs}.values()
    

In [41]:
unique_docs = {doc.metadata['_id']: doc for doc in combined_docs}.values()
print(len(combined_docs))
for doc in docs:
    print("===========================")
    print(doc.metadata.get('_id'))
    print(doc.page_content)

6
None
Exhibit 4.5
SUPPLY AGREEMENT
between
PROFOUND MEDICAL INC.
and
PHILIPS MEDICAL SYSTEMS NEDERLAND B.V.
None
THIS AGREEMENT is made July 31, 2017
BETWEEN:
PROFOUND MEDICAL INC., a company incorporated under the laws of the province of Ontario and having its
registered address at 2400 Skymark, Unit 6, Mississauga, Ontario L4W 5K5, Canada
(hereinafter referred to as “Customer”)
- and -
PHILIPS MEDICAL SYSTEMS NEDERLAND B.V., a company
incorporated under the laws of the Netherlands with its principal place of business at Veenpluis 4-6 5684 PC Best, the
Netherlands
(hereinafter referred to as “Philips”)
Customer and Philips hereinafter also collectively referred to as the “Parties” and individually as a “Party”.
WHEREAS:
A.
Pursuant to the Asset and Share Purchase Agreement (the “Purchase Agreement”) entered into on June 30, 2017 by Customer,
Koninklijke Philips NV (“Philips NV”) N.V. and Customer agreed to execute and deliver (or cause to be executed and delivered) certain
ancillary 

In [ ]:
import json
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS, Qdrant
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from LangchainGeminiLoader import get_llm
from langchain.prompts import PromptTemplate
import os, shutil
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
import re
import codecs

def clean_text_with_ftfy(text: str) -> str:
    try:
        # Decode literal escape sequences like \\u00a0 or \\xa0 to actual chars
        text = codecs.decode(text, 'utf-8')
    except Exception:
        pass

    # Replace invisible Unicode whitespace chars with a regular space
    text = text.replace('\u00a0', ' ')  # Non-breaking space
    text = text.replace('\xa0', ' ')    # Also non-breaking space in hex
    text = text.replace('\u200b', '')   # Zero-width space
    text = text.replace('\u202f', ' ')  # Narrow no-break space
    text = text.replace('\u2009', ' ')  # Thin space

    # Optional: Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text).strip()
    return text
if __name__ == '__main__':
    folder = 'output/'
    
    total_token_count = 0
    total_tokens_of_pdfs = 0
    # for filename in os.listdir(folder):
    #     file_path = os.path.join(folder, filename)
    #     try:
    #         if os.path.isfile(file_path) or os.path.islink(file_path):
    #             os.unlink(file_path)
    #     except Exception as e:
    #         print('Failed to delete %s. Reason: %s' % (file_path, e))

    pdfs = list(filter(lambda x: x.endswith('.pdf'), os.listdir('contract_pdfs')))
    # pdfs = ['BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.pdf']
    # pdfs = [
    #     'BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.pdf', 
    #     'FUSIONPHARMACEUTICALSINC_06_05_2020-EX-10.17-Supply Agreement - FUSION.pdf',
    #     'MEDIWOUNDLTD_01_15_2014-EX-10.6-SUPPLY AGREEMENT copy.pdf',
    #     'FreezeTagInc.pdf'
    # ]
    pdfs = ['MEDIWOUNDLTD_01_15_2014-EX-10.6-SUPPLY AGREEMENT.pdf']
    for pdf_name in pdfs:
        print("Loop")
        loader = PyMuPDFLoader(f'contract_pdfs/{pdf_name}')

        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=300,
            length_function=len
        )

        docs = splitter.split_documents(docs)
        token_of_doc=0
        for doc in docs:
            token_of_doc += len(doc.page_content)/4
        total_tokens_of_pdfs += token_of_doc

        embeddings = SentenceTransformerEmbeddings(
            model_name="all-MiniLM-L6-v2"
        )

        # vectorstore = FAISS.from_documents(
        #     documents=docs,
        #     embedding=embeddings,
        # )
        qdrant = Qdrant.from_documents(
            documents=docs,
            embedding=embeddings,
            url="http://localhost:6333",  # Local Qdrant instance
            prefer_grpc=False
        )
        
        # qdrant.similarity_search(query="Contact Information of Sponsors and Orgnaizations")
        retriever = qdrant.as_retriever(search_type="mmr", search_kwargs={'k': 3})
        # query = "Find - Email Contact By Address Tel Fax Signature Phn Name Sponsors Organization Club"
        contact_info_docs = retriever.get_relevant_documents(query="IN WITNESS WHEREOF both parties have agreed to")
        transaction_docs = retriever.get_relevant_documents(query="The party has agreed to deliver the following amount XXX dollars $ or YY % to the Club for the following quarter/month period")
        txt = ''
        for doc in contact_info_docs + transaction_docs:
            txt += doc.page_content
        txt = clean_text_with_ftfy(txt)
        prompt = PromptTemplate.from_template("""
        You are an contract analyzing assistant.
        1. You are required to extract the following data from the given Context.
        - Orgnaizations Involved.
        - Contract Agreement Summary.
        - Compliance and Expectations Summary.
        - Amount Deliverables for shipments in Agreement context.
        - Contract Start & end dates.
        - Termination & Expiry Information Summary.
        2. Do the following corrections
        - If the context has [***] USD or percentage or contact information. You can replace it with some random Amount Value or Percentage or Contact Information.
        - Then you can analyze to extract the information.
        Context: {context}

        **Output Format**
        1. Format the response in the json format in given schema.
        ### JSON Schema
        {{
            "organizations_involved": [
                {{
                "name": "Organization Name",
                "email": "contact@example.com",
                "phone": "+1-555-123-4567"
                }}
            ],
            "contract_agreement_summary": [
                "Summarize the agreement in short bullet points.",
            ],
            "compliance_and_expectations_summary": [
                "Summarize all compliance obligations and expectations in separate statements.",
            ],
            "amount_deliverables_for_shipments": [
                {{
                "amount": "e.g. 1000 units",
                "period": "e.g. Monthly",
                "description": "Describe what is being delivered and under what terms."
                }}
            ],
            "contract_start_date": "YYYY-MM-DD",
            "contract_end_date": "YYYY-MM-DD",
            "termination_and_expiry_summary": [
                "List the key conditions or clauses under which the contract can be terminated.",
                "Include expiry rules or notice periods if stated."
            ]
        }}
        # Note
        - Output only a json object. Never return any code block or any other format.
        - Don't add any marker like ```json, ``` etc
        - Don't output any extra character other than proper json
        - Ensure all variations of duplicate data are preserved with contextual clarity.
        - The merging process must account for nested structures and hierarchy.
        - Further processing includes building knowledge graph from this output json. Hence the node and relationship is important
        """)
        llm = get_llm()
        
        prompt = prompt.invoke({"context": txt})
        token_count = len(prompt.to_string())/4
        print("Invoking LLM")
        resp = llm.invoke(prompt)

        print(f'Token Count: {resp.usage_metadata.get('total_tokens')}')
        print(f'Response: {resp.content}')
        
        
        resp_json = json.loads(resp.content.strip('```').strip('json'))
        json_obj = {
            "total_token": resp.usage_metadata.get('total_tokens'),
            "file_name": pdf_name,
        }
        
        with open(f'output/{pdf_name.split(".")[0]}.json', 'w') as json_file:
            json.dump(json_obj, json_file, indent=4)
        with open('output/'+pdf_name.split('.')[0]+'.txt', 'w') as f:
            f.write(f'Token Count: {resp.usage_metadata.get('total_tokens')} \n Context: {txt} \n Response: {resp.content}')
        
        # retriever = vectorstore.as_retriever(search_type="mmr",
        #     search_kwargs={'k': 8})
        # model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
        # reranker = CrossEncoderReranker(model=model, top_n=3)
        # compression_retriever = ContextualCompressionRetriever(
        #     base_compressor=reranker,
        #     base_retriever=retriever
        # )
        

        # query = "Organization decides to pay XXX dollars as per agreement for the shipment"
        # query = "Execution Clause and Contact Information"
        # rel_docs = retriever.get_relevant_documents(query=query)
        # query="Contact Information"
        # query = "Email Address Tel Fax Signature Phn Name Sponsors Organization"
        # query="IN WITNESS WHEREOF"

        # reranked_docs = compression_retriever.get_relevant_documents(query="Email Contact By Address Tel Fax Signature Phn Name Sponsors Organization Club")



        # with open('output/'+pdf_name.split('.')[0]+'.txt', 'w') as f:
        #     f.write('\n'.join([doc.page_content for doc in contact_info_docs + transaction_docs]))

Loop
Invoking LLM
Token Count: 3134
Response: ```json
{
    "organizations_involved": [
        {
            "name": "Challenge Bioproducts Corporation, Ltd. (CBC)",
            "email": "cbc@email.com",
            "phone": "+55-5572-045"
        },
        {
            "name": "MediWound Ltd.",
            "email": "mediwound@email.com",
            "phone": "+972 8 932 4010"
        },
        {
            "name": "Golden Life International Co., Ltd.",
            "email": null,
            "phone": null
        }
    ],
    "contract_agreement_summary": [
        "Agreement between CBC and MediWound for the supply of Bromelain SP.",
        "CBC and MediWound are independent contractors.",
        "The agreement supersedes all prior agreements except for the TT Agreement until the Effective Date, after which the MOU is also superseded.",
        "Terms and conditions of the original agreement remain in effect unless explicitly amended.",
        "The agreement is governed by the

In [ ]:
import re
import codecs

def clean_text_with_ftfy(text: str) -> str:
    try:
        # Decode literal escape sequences like \\u00a0 or \\xa0 to actual chars
        text = codecs.decode(text, 'utf-8')
    except Exception:
        pass

    # Replace invisible Unicode whitespace chars with a regular space
    text = text.replace('\u00a0', ' ')  # Non-breaking space
    text = text.replace('\xa0', ' ')    # Also non-breaking space in hex
    text = text.replace('\u200b', '')   # Zero-width space
    text = text.replace('\u202f', ' ')  # Narrow no-break space
    text = text.replace('\u2009', ' ')  # Thin space

    # Optional: Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text).strip()
    return text
clean_text_with_ftfy(txt)

'procure that its Affiliates and Sub-Contractors comply fully with the provision of this Agreement in connection with such performance. 13. Miscellaneous 13.1 Failure or delay by either party in exercising or enforcing any right or remedy under this Agreement in whole or in part shall not be deemed a waiver thereof or prevent the subsequent exercise of that or any other rights or remedy. 13.2 CBC and its employees and MediWound and its employees shall at all times be considered as independent contractors of each other, and at no time or under any circumstances shall they be considered employees, representatives, partners or agents of each other. 13.3 This Agreement shall constitute the entire agreement and understanding of the parties relating to the subject matter of this Agreement and supersede all prior oral or written agreements, understandings or arrangements between them relating to such subject, except for the TT Agreement. The MOU shall be deemed so superseded by this Agreement